In [2]:
import cygnet
import torch, time

test_fixed_precision = False

def total_loss(pred, target):
    return cygnet.focal_loss(pred, target)

cygnet.start_timer()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
torch.set_default_device(device)

b_size = 8
num_batches = 2
epochs = 5

H = 2304 // 4
W = 4096 // 4

data = [torch.randn(b_size, 1, H, W) for _ in range(num_batches)]
targets = [torch.randn(b_size, 1, H, W) for _ in range(num_batches)]
dummy = torch.randn(1, 1, H, W)

cygnet.end_timer_and_print("setup")
print("\n=======\n")
cygnet.start_timer()

net = cygnet.Net(32)
opt = torch.optim.SGD(net.parameters(), lr=0.001)
scaler = torch.amp.GradScaler("cuda", enabled=True)

print(net)

b = 0
for epoch in range(epochs):
    print(f"start epoch {epoch}:")
    for input, target in zip(data, targets):
        print(f"batch #{b}")
        b += 1
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=True):
            output = net(input)
            loss = total_loss(output, target)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()
cygnet.end_timer_and_print("Mixed precision:")

net.eval()
# Warmup
for _ in range(10):
    _ = net(dummy)

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16, enabled=True):
        _ = net(dummy)
torch.cuda.synchronize()
tim = (time.perf_counter() - t0) / 10 * 1000
print(f"in inference: {tim:.1f} ms per frame")

Using cuda device

setup
Total execution time = 0.006 sec
Max memory used by tensors = 0.163 Gbytes


Net(
  (bn_input): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (dc1): DoubleDown2(
    (seq): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU()
    )
    (mp): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (dc2): DoubleDown2(
    (seq): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=Tr